# Prompt Evaluation & Testing 🧪📊

In traditional software engineering, testing is deterministic: you write a unit test (assert add(2, 2) == 4) and your CI/CD pipeline passes or fails. In GenAI, prompts are probabilistic. If you tweak a single word in a prompt template, output wording shifts, and a hardcoded test string will break.

Prompt evaluation brings Test-Driven Development (TDD) and CI/CD rigor to AI applications.

## Phase 1: The Traditional Testing Dilemma vs. GenAI

**Traditional Code:** Deterministic. Same input $\rightarrow$ exact same binary or string output. Unit tests check for exact equality.

**GenAI Code:** Probabilistic. Even with low temperature, minor layout or vocabulary fluctuations happen. Checking string equality (output == "expected") fails constantly due to harmless variations in phrasing.

To test prompts reliably, you must shift from exact-match assertions to semantic evaluation.

## Phase 2: The Three Pillars of Prompt Evaluation
To build a robust evaluation pipeline for your prompts, you need three components:

**The Golden Dataset:** A curated dataset of 50 to 200 representative production inputs (test cases) paired with expected behaviors, edge cases, and constraints.

**The Test Runner:** A tool or script that loops through your golden dataset, feeds each input into your prompt template, and captures the LLM's output.

**The Grader (Assertions):** Automated logic that scores whether the output successfully met your criteria.

## Phase 3: Evaluation Methods (How We Grade Outputs)
Depending on what your prompt is doing, you use different types of automated assertions:

### 1. Deterministic Assertions (Code-Based)
**Regex / JSON Schema Validation:** Does the output parse cleanly into your Pydantic or Zod schema? (If it fails to parse, the test fails instantly).

**Substring / Keyword Checks:** Does a customer support summary contain mandatory tracking codes or exclude forbidden phrases?

### 2. LLM-as-a-Judge (Semantic Assertions)
For nuanced text generation (like code reviews, creative text, or conversational summaries), code-based checks aren't enough. You use a more powerful model (e.g., GPT-4o or Claude 3.5 Sonnet) as an automated grader.

**How it works:** You send the user input and the model's output to a judge prompt with a strict rubric: 

 "Evaluate the following code review on a scale of 1 to 5 for technical accuracy. Output a JSON object with score and reasoning."

## Phase 4: Developer Workflow & Tools
In modern production environments, developers don't build custom evaluation loops from scratch. They rely on standard developer-first frameworks:

Promptfoo: A popular open-source CLI tool that lets you define prompts and test cases in a simple YAML file, run local matrix evaluations, and wire checks directly into GitHub Actions CI gates.
LangSmith / Braintrust / DeepEval: Platforms that track prompt versions, log test runs, measure latency/cost regressions, and monitor live production evaluation traces.  

## Phase 5: Example Test Configuration (Promptfoo YAML Style)

Here is what an evaluation test suite looks like in practice. Instead of writing heavy testing code, you declare your test cases in a configuration file:

In [ ]:
# promptfoo config example
prompts:
  - "file://prompts/code_reviewer.txt"

providers:
  - openai:gpt-4.1-mini

tests:
  - description: "Catch insecure eval injection"
    vars:
      source_code: "eval(user_input)"
    assert:
      - type: contains
        value: "security vulnerability"
      - type: javascript
        value: output.length < 500  # Ensure response is concise

  - description: "Handle clean utility code safely"
    vars:
      source_code: "def add(a, b): return a + b"
    assert:
      - type: llm-rubric
        value: "Confirm the model states that this code snippet is safe and clean."